<a href="https://colab.research.google.com/github/tramthanh1025/Try/blob/main/Datathon_Analysis_Nerissa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modeling goal

Defining users persona. I've found that all of you have done an extremely great process on the model improvement so I want to try to see if I can add more info by the clustering method on the sample Arthur sent. The original dataset is too large for this method.

## Connect DuckDB

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q duckdb

In [3]:
import duckdb

con = duckdb.connect()

train_path = "/content/drive/MyDrive/Datathon/train_rsample_100k.parquet"

con.execute(f"""
    SELECT COUNT(*)
    FROM read_parquet('{train_path}')
""").fetchone()

(100000,)

In [4]:
con.execute(f"""
CREATE OR REPLACE VIEW train AS
SELECT * FROM read_parquet('{train_path}');
""")

# Initial EDA

Similar EDA process for the original training set. I just want to see the differences.

In [ ]:
con.execute("DESCRIBE train").df()

In [ ]:
con.execute("SELECT * FROM train LIMIT 15").df()

**Data Exploration: Global Reward Rate**

In [7]:
baseline_stats = con.execute("""
    SELECT
        COUNT(*) AS total_notifications,
        SUM(CASE WHEN session_end_completed THEN 1 ELSE 0 END) AS positive_samples,
        SUM(CASE WHEN NOT session_end_completed THEN 1 ELSE 0 END) AS negative_samples,
        AVG(CASE WHEN session_end_completed THEN 1 ELSE 0 END) AS global_reward_rate
    FROM train
""").df()

# Extract values for easier use
total = baseline_stats['total_notifications'][0]
pos = baseline_stats['positive_samples'][0]
neg = baseline_stats['negative_samples'][0]
rate = baseline_stats['global_reward_rate'][0]

print(f"Total Notifications: {total:,}")
print(f"Positive Interactions (Success): {pos:,}")
print(f"Negative Interactions (Ignored): {neg:,}")
print(f"Global Reward Rate: {rate:.2%}")

Total Notifications: 100,000
Positive Interactions (Success): 14,704.0
Negative Interactions (Ignored): 85,296.0
Global Reward Rate: 14.70%


=> Result: The global reward rate on the sample set is 14.7%, quite close to the original training set.

**Data Exploration: Language and Template Analysis**

In [8]:
distinct_languages_df = con.execute("SELECT DISTINCT ui_language FROM train ORDER BY ui_language").df()
distinct_languages = distinct_languages_df['ui_language'].tolist() # Prepare for further analysis later

print(f"Number of UI Languages: {len(distinct_languages)}")
print(f"Distinct UI Languages: {distinct_languages}")

Number of UI Languages: 23
Distinct UI Languages: ['ar', 'cs', 'de', 'dn', 'el', 'en', 'es', 'fr', 'hi', 'hu', 'id', 'it', 'ja', 'ko', 'pl', 'pt', 'ro', 'ru', 'th', 'tr', 'uk', 'vi', 'zs']


=> Result: 23 distinct languages. 2 languages less than the original one, which is bn and ta, Bengali and Tamil from GG search. Both are quite uncommon languages so I think we can accept this.




In [9]:
distinct_eligible_templates_df = con.execute("SELECT DISTINCT eligible_templates FROM train ORDER BY eligible_templates").df()
distinct_eligible_templates = distinct_eligible_templates_df['eligible_templates'].tolist() # Prepare for further analysis later

print(f"Number of Eligible Templates: {len(distinct_eligible_templates)}")
print(f"Distinct Eligible Templates set:")
print(distinct_eligible_templates_df)

Number of Eligible Templates: 8
Distinct Eligible Templates set:
               eligible_templates
0                       [A, K, H]
1                             [C]
2  [G, E, B, A, K, H, J, L, F, D]
3     [G, E, B, K, H, J, L, F, D]
4                          [K, H]
5                       [K, H, A]
6     [K, H, G, E, B, J, L, F, D]
7  [K, H, G, E, B, J, L, F, D, A]


=> Result: 8 distinct eligible templates in which only one class of users eligible to receive template C and 4 classes of users eligible to us template A.

The missing one is [G, E, B, A].

In [10]:
template_performance_df = con.execute("""
    SELECT
        selected_template,
        COUNT(*) AS notification_count,
        AVG(CASE WHEN session_end_completed THEN 1.0 ELSE 0.0 END) AS success_rate
    FROM train
    GROUP BY selected_template
    ORDER BY notification_count DESC
""").df()

print(template_performance_df)

   selected_template  notification_count  success_rate
0                  E               10422      0.131549
1                  K               10395      0.136893
2                  B               10342      0.131213
3                  G               10332      0.133662
4                  F               10309      0.129887
5                  D               10303      0.134039
6                  L               10267      0.134606
7                  H               10239      0.133411
8                  J               10236      0.128859
9                  A                4053      0.272144
10                 C                3102      0.413282


In [11]:
sorted_by_success_rate_df = template_performance_df.sort_values(by='success_rate', ascending=False)
print("Templates sorted by success rate (top 5):")
print(sorted_by_success_rate_df.head())

print("\nTemplates sorted by success rate (bottom 5):")
print(sorted_by_success_rate_df.tail())

Templates sorted by success rate (top 5):
   selected_template  notification_count  success_rate
10                 C                3102      0.413282
9                  A                4053      0.272144
1                  K               10395      0.136893
6                  L               10267      0.134606
5                  D               10303      0.134039

Templates sorted by success rate (bottom 5):
  selected_template  notification_count  success_rate
7                 H               10239      0.133411
0                 E               10422      0.131549
2                 B               10342      0.131213
4                 F               10309      0.129887
8                 J               10236      0.128859


=> Result: C and A are still dominators. The order of the rest is a bit different but they're all around 13% just like in the original set.

**Data Exploration: Time zone and Hours of the day**:


In [12]:
timezone_mappings = {
    'UTC+1 (Western/Central Europe)': (1, ['de', 'fr', 'it', 'es', 'nl', 'pl', 'cs', 'el', 'hu', 'dn']),
    'UTC+0 (UK/Western Europe)': (0, ['en', 'pt']),
    'UTC+2 (Eastern Europe/Middle East)': (2, ['ru', 'uk', 'ro', 'ar', 'tr']),
    'UTC+5.5 (India)': (5, ['hi']),
    'UTC+7 (Southeast Asia)': (7, ['id', 'th', 'vi']),
    'UTC+9 (East Asia)': (9, ['ja', 'ko']),
    'Unknown/Other': (0, ['zs']) # 'zs' is less common code, assigned to UTC+0 as a default.
}

# Ensure all distinct_languages are covered
all_mapped_languages = set()
for offset, languages in timezone_mappings.values():
    all_mapped_languages.update(languages)

# Check for any unmapped languages from the distinct_languages list
# distinct_languages is from the kernel state, which is: ['ar', 'bn', 'cs', 'de', 'dn', 'el', 'en', 'es', 'fr', 'hi', 'hu', 'id', 'it', 'ja', 'ko', 'pl', 'pt', 'ro', 'ru', 'ta', 'th', 'tr', 'uk', 'vi', 'zs']

unmapped_languages = [lang for lang in distinct_languages if lang not in all_mapped_languages]

# Add any remaining unmapped languages to the 'Unknown/Other' category if they exist
if unmapped_languages:
    print(f"Adding unmapped languages to 'Unknown/Other': {unmapped_languages}")
    # Convert the tuple to a list to modify, then convert back to tuple
    unknown_offset, unknown_langs = timezone_mappings['Unknown/Other']
    timezone_mappings['Unknown/Other'] = (unknown_offset, sorted(list(set(unknown_langs + unmapped_languages))))


print("Timezone Mappings Dictionary created:")
for tz_name, (offset, langs) in timezone_mappings.items():
    print(f"  {tz_name}: Offset={offset}, Languages={langs}")

Timezone Mappings Dictionary created:
  UTC+1 (Western/Central Europe): Offset=1, Languages=['de', 'fr', 'it', 'es', 'nl', 'pl', 'cs', 'el', 'hu', 'dn']
  UTC+0 (UK/Western Europe): Offset=0, Languages=['en', 'pt']
  UTC+2 (Eastern Europe/Middle East): Offset=2, Languages=['ru', 'uk', 'ro', 'ar', 'tr']
  UTC+5.5 (India): Offset=5, Languages=['hi']
  UTC+7 (Southeast Asia): Offset=7, Languages=['id', 'th', 'vi']
  UTC+9 (East Asia): Offset=9, Languages=['ja', 'ko']
  Unknown/Other: Offset=0, Languages=['zs']


In [13]:
hourly_reward_rates_by_timezone = {}

for tz_name, (offset, languages) in timezone_mappings.items():
    languages_str = ", ".join(f"'{lang}'" for lang in languages)

    query = f"""
        SELECT
            CAST(MOD(datetime * 24 + {offset}, 24) AS INTEGER) AS local_hour,
            COUNT(*) AS n_notifications,
            AVG(CASE WHEN session_end_completed THEN 1.0 ELSE 0.0 END) AS reward_rate
        FROM train
        WHERE ui_language IN ({languages_str})
        GROUP BY local_hour
        ORDER BY local_hour
    """

    print(f"Processing timezone: {tz_name} (Languages: {languages})")

    hourly_reward_rates_by_timezone[tz_name] = con.execute(query).df()
    print(f"  - Data for {tz_name} loaded. {len(hourly_reward_rates_by_timezone[tz_name])} rows.\n")

print("Hourly reward rates calculated for all timezones.")

Processing timezone: UTC+1 (Western/Central Europe) (Languages: ['de', 'fr', 'it', 'es', 'nl', 'pl', 'cs', 'el', 'hu', 'dn'])
  - Data for UTC+1 (Western/Central Europe) loaded. 25 rows.

Processing timezone: UTC+0 (UK/Western Europe) (Languages: ['en', 'pt'])
  - Data for UTC+0 (UK/Western Europe) loaded. 25 rows.

Processing timezone: UTC+2 (Eastern Europe/Middle East) (Languages: ['ru', 'uk', 'ro', 'ar', 'tr'])
  - Data for UTC+2 (Eastern Europe/Middle East) loaded. 25 rows.

Processing timezone: UTC+5.5 (India) (Languages: ['hi'])
  - Data for UTC+5.5 (India) loaded. 24 rows.

Processing timezone: UTC+7 (Southeast Asia) (Languages: ['id', 'th', 'vi'])
  - Data for UTC+7 (Southeast Asia) loaded. 25 rows.

Processing timezone: UTC+9 (East Asia) (Languages: ['ja', 'ko'])
  - Data for UTC+9 (East Asia) loaded. 25 rows.

Processing timezone: Unknown/Other (Languages: ['zs'])
  - Data for Unknown/Other loaded. 25 rows.

Hourly reward rates calculated for all timezones.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Calculate the number of subplots needed based on the number of timezones
num_timezones = len(hourly_reward_rates_by_timezone)
num_cols = 2  # Number of columns for subplots
num_rows = int(np.ceil(num_timezones / num_cols))

plt.figure(figsize=(15, 6 * num_rows))

for i, (tz_name, df) in enumerate(hourly_reward_rates_by_timezone.items()):
    plt.subplot(num_rows, num_cols, i + 1)
    plt.plot(df['local_hour'], df['reward_rate'], marker='o', linestyle='-')
    plt.title(f'Hourly Reward Rate for {tz_name}')
    plt.xlabel('Local Hour of Day (0-23)')
    plt.ylabel('Average Reward Rate')
    plt.xticks(np.arange(0, 24, 2)) # Show every other hour for clarity
    plt.grid(True)
    plt.ylim(0, df['reward_rate'].max() * 1.2) # Set y-limit dynamically based on data

plt.tight_layout()
plt.show()


=> Result: The shapes of all are quite similar to the original ones, but more fluctuated (smaller dataset). Only the India timezone shows the biggest difference (lack of 2 languages). The Southeast Asia and Easten Europe/Middle East also show a change in first and second peak.

# Feature Engineering

I add some columns for easier clustering and interpretation.

In [15]:
import pandas as pd

# Load the necessary columns from the DuckDB 'train' view into a Pandas DataFrame
df = con.execute("""
    SELECT
        datetime,
        ui_language,
        CAST(session_end_completed AS INTEGER) AS session_end_completed, -- Ensure 0/1 encoding
        selected_template,
        eligible_templates,
        n_eligible
    FROM train
""").df()

# Create lookup dictionaries
timezone_group_map = {}
timezone_offset_map = {}
for tz_name, (offset, languages) in timezone_mappings.items():
    for lang in languages:
        timezone_group_map[lang] = tz_name
        timezone_offset_map[lang] = offset

# Create new columns timezone_group and timezone_offset
df['timezone_group'] = df['ui_language'].map(timezone_group_map)
df['timezone_offset'] = df['ui_language'].map(timezone_offset_map)

# Calculate the local_hour column
df['local_hour'] = ((df['datetime'] * 24 + df['timezone_offset']) % 24).astype(int)

# Create a time_of_day column
bins = [-0.1, 6, 12, 18, 24] # 0-5 for Night, 6-11 for Morning, 12-17 for Afternoon, 18-23 for Evening.
labels = ['Night', 'Morning', 'Afternoon', 'Evening']
df['time_of_day'] = pd.cut(df['local_hour'], bins=bins, labels=labels, right=False, include_lowest=True)

# Display the first 5 rows of the df DataFrame
print("First 5 rows of the DataFrame with new columns:")
print(df[['datetime', 'ui_language', 'eligible_templates', 'timezone_group', 'local_hour', 'time_of_day', 'session_end_completed', 'n_eligible']].head())

# Print the value counts for the timezone_group column
print("\nValue counts for timezone_group:")
print(df['timezone_group'].value_counts())

# Print the value counts for the time_of_day column
print("\nValue counts for time_of_day:")
print(df['time_of_day'].value_counts())

First 5 rows of the DataFrame with new columns:
   datetime ui_language              eligible_templates  \
0  9.283380          fr     [K, H, G, E, B, J, L, F, D]   
1  9.623044          es     [K, H, G, E, B, J, L, F, D]   
2  5.172986          de  [G, E, B, A, K, H, J, L, F, D]   
3  6.269595          en     [G, E, B, K, H, J, L, F, D]   
4  0.913009          es     [G, E, B, K, H, J, L, F, D]   

                   timezone_group  local_hour time_of_day  \
0  UTC+1 (Western/Central Europe)           7     Morning   
1  UTC+1 (Western/Central Europe)          15   Afternoon   
2  UTC+1 (Western/Central Europe)           5       Night   
3       UTC+0 (UK/Western Europe)           6     Morning   
4  UTC+1 (Western/Central Europe)          22     Evening   

   session_end_completed  n_eligible  
0                      0           9  
1                      0           9  
2                      0          10  
3                      0           9  
4                      0           

## Modelling

**Data Preprocessing and Scaling**

It's easiest to proceed clustering by numerical features so I used the n_eligible (differentiate the C users); local_hour and time_of_day_encoded (users' habit); timezone_offset and timezone_group_encoded (users' nationality or residency).

In [16]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Initialize LabelEncoders
le_timezone_group = LabelEncoder()
le_time_of_day = LabelEncoder()

# Apply Label Encoding to categorical features
df['timezone_group_encoded'] = le_timezone_group.fit_transform(df['timezone_group'])
df['time_of_day_encoded'] = le_time_of_day.fit_transform(df['time_of_day'])

# From the df DataFrame, select the following numerical features for clustering
numerical_features = [
    'n_eligible',
    'local_hour',
    'timezone_offset',
    'timezone_group_encoded',
    'time_of_day_encoded'
]

X = df[numerical_features].values

# Instantiate StandardScaler and fit it to the selected numerical features (X).
scaler = StandardScaler()
scaler.fit(X)

# Transform the numerical features using the fitted scaler and store the result in X_scaled.
X_scaled = scaler.transform(X)

print(f"Shape of original numerical features (X): {X.shape}")
print(f"Shape of scaled numerical features (X_scaled): {X_scaled.shape}")
print("First 5 rows of X_scaled (after scaling):\n", X_scaled[:5])

Shape of original numerical features (X): (100000, 5)
Shape of scaled numerical features (X_scaled): (100000, 5)
First 5 rows of X_scaled (after scaling):
 [[-0.08967704 -0.95141569  0.12485483  0.16667783  0.75059067]
 [-0.08967704  0.33470048  0.12485483  0.16667783 -1.12518259]
 [ 0.54773175 -1.27294473  0.12485483  0.16667783  1.6884773 ]
 [-0.08967704 -1.11218021 -0.54181981 -0.66813051  0.75059067]
 [-0.08967704  1.46005212  0.12485483  0.16667783 -0.18729596]]


**Determining optimal numbers of cluster by Elbow Method**

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

k_range = range(1, 11)
inertia = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)  # n_init to suppress warning
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(10, 6))
plt.plot(k_range, inertia, marker='o')
plt.title('Elbow Method for Optimal K')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.xticks(k_range)
plt.grid(True)
plt.show()

print("Inertia values for each K:", inertia)

=> Around 4-5 clusters. I choose 4.

In [18]:
n_clusters = 4

kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

df['cluster'] = cluster_labels

print(f"K-Means clustering performed with {n_clusters} clusters.")
print("First 5 rows of DataFrame with cluster labels:")
print(df[['datetime', 'ui_language', 'selected_template', 'local_hour', 'session_end_completed', 'cluster']].head())

K-Means clustering performed with 4 clusters.
First 5 rows of DataFrame with cluster labels:
   datetime ui_language selected_template  local_hour  session_end_completed  \
0  9.283380          fr                 L           7                      0   
1  9.623044          es                 D          15                      0   
2  5.172986          de                 K           5                      0   
3  6.269595          en                 G           6                      0   
4  0.913009          es                 L          22                      0   

   cluster  
0        2  
1        0  
2        2  
3        2  
4        0  


# Clusters Analysis

**Overall Analysis**

In [19]:
print("Cluster Sizes:")
print(df['cluster'].value_counts().sort_index())

print("\nCluster-wise session_end_completed Rate:")
print(df.groupby('cluster')['session_end_completed'].mean().sort_index())

print("\nDescriptive Statistics for each Cluster:")
cluster_summary = df.groupby('cluster')[numerical_features + ['session_end_completed']].mean()
print(cluster_summary)

Cluster Sizes:
cluster
0    55765
1     5573
2    35399
3     3263
Name: count, dtype: int64

Cluster-wise session_end_completed Rate:
cluster
0    0.139944
1    0.135654
2    0.136275
3    0.404536
Name: session_end_completed, dtype: float64

Descriptive Statistics for each Cluster:
         n_eligible  local_hour  timezone_offset  timezone_group_encoded  \
cluster                                                                    
0          9.406527   17.173962         0.560477                0.560477   
1          9.296968   11.213530         5.064777                4.835457   
2          9.437300    6.543744         0.553631                0.553631   
3          1.112780   12.248238         0.672081                0.684340   

         time_of_day_encoded  session_end_completed  
cluster                                              
0                   0.443683               0.139944  
1                   1.330702               0.135654  
2                   2.360745              

=> Biggest cluster: Cluster 3. I predict this is template C users.

First impression:


*   Cluster 0 has highest local hour => may be afternoon - evening users
*   Cluster 1 has most different timezone_offset => Indian/Asian users
*   Cluster 2 has lowest local hour => may be night owls or early birds
*   Cluster 3 has lowest n_eligible => template C user




**Detailed Analysis**

In [20]:
import numpy as np

# Calculate comprehensive descriptive statistics for numerical features and 'session_end_completed'
# Define the features for which to calculate statistics
features_for_stats = numerical_features + ['session_end_completed']

# Calculate mean, median, and standard deviation for each cluster
cluster_means = df.groupby('cluster')[features_for_stats].mean()
cluster_medians = df.groupby('cluster')[features_for_stats].median()
cluster_stds = df.groupby('cluster')[features_for_stats].std()

# Combine these into a single, more readable DataFrame (optional, but good for presentation)
cluster_descriptive_stats = pd.concat(
    [cluster_means.add_prefix('mean_'),
     cluster_medians.add_prefix('median_'),
     cluster_stds.add_prefix('std_')],
    axis=1
)

print("\nComprehensive Descriptive Statistics for each Cluster:")
print(cluster_descriptive_stats)



Comprehensive Descriptive Statistics for each Cluster:
         mean_n_eligible  mean_local_hour  mean_timezone_offset  \
cluster                                                           
0               9.406527        17.173962              0.560477   
1               9.296968        11.213530              5.064777   
2               9.437300         6.543744              0.553631   
3               1.112780        12.248238              0.672081   

         mean_timezone_group_encoded  mean_time_of_day_encoded  \
cluster                                                          
0                           0.560477                  0.443683   
1                           4.835457                  1.330702   
2                           0.553631                  2.360745   
3                           0.684340                  1.300644   

         mean_session_end_completed  median_n_eligible  median_local_hour  \
cluster                                                            

In [21]:
categorical_features = ['ui_language', 'selected_template', 'timezone_group', 'time_of_day']

for feature in categorical_features:
    print(f"\nDistribution of '{feature}' within each Cluster:")
    # Value counts
    cluster_category_counts = df.groupby('cluster')[feature].value_counts()
    print("Value Counts:")
    print(cluster_category_counts)

    # Proportions
    cluster_category_proportions = df.groupby('cluster')[feature].value_counts(normalize=True)
    print("Proportions:")
    print(cluster_category_proportions)



Distribution of 'ui_language' within each Cluster:
Value Counts:
cluster  ui_language
0        en             22250
         es             17771
         pt              6352
         ru              1965
         fr              1927
         de              1299
         ar              1014
         it               838
         tr               712
         pl               510
         cs               311
         ro               258
         hu               234
         uk               143
         dn               137
         el                44
1        zs              1827
         vi              1519
         ja               966
         id               430
         ko               353
         hi               317
         th               161
2        en             17156
         es              5783
         pt              2855
         ru              2344
         fr              1786
         de              1438
         it               857
         ar  

### Cluster Analysis Summary

Based on the K-Means clustering with K=4, the following profiles have been identified for each cluster:

**Overall Global Reward Rate:** 14.70%

---

#### Cluster 0: Afternoon Engagers (Size: 55,765 samples)

*   **Key Characteristics:**
    *   **Average `n_eligible`:** ~9.41 (High number of eligible templates)
    *   **Average `local_hour`:** ~17.17 (Late Afternoon/Evening in local time)
    *   **Average `timezone_offset`:** ~0.56 (Slightly positive offset from UTC)
    *   **`session_end_completed` Rate:** **13.99%** (Slightly below global average)

*   **Categorical Distribution:**
    *   **`time_of_day_categorical`:** Predominantly Afternoon (55.63%) and Evening (44.37%) in local time. This suggests that notifications are being sent during users' active late-day hours.
    *   **`ui_language` & `timezone_group`:** Heavily represented by UTC+0 (UK/Western Europe - ~50%) and UTC+1 (Western/Central Europe - ~35%) language groups, primarily `en` and `es`.
    *   **`selected_template`:** All templates (B, D, E, F, G, H, J, K, L) are distributed fairly evenly, with 'A' having a lower proportion and 'C' having almost no presence.

*   **Insights:** This is the largest cluster, primarily from Western/Central European timezones, engaging in their local afternoons and evenings. Their completion rate is close to the global average. To improve engagement, consider A/B testing different templates during these peak local activity hours, focusing on languages prevalent in these timezones.

---

#### Cluster 1: East Asian/Southeast Asian Users (Size: 5,573 samples)

*   **Key Characteristics:**
    *   **Average `n_eligible`:** ~9.30 (High number of eligible templates)
    *   **Average `local_hour`:** ~11.21 (Late Morning/Midday in local time)
    *   **Average `timezone_offset`:** ~5.06 (Higher positive offset from UTC)
    *   **`session_end_completed` Rate:** **13.57%** (Lower than global average)

*   **Categorical Distribution:**
    *   **`time_of_day_categorical`:** Fairly distributed, with Afternoon (38.65%), Morning (27.76%), and Night (21.98%). This indicates activity throughout different local time periods.
    *   **`ui_language` & `timezone_group`:** Heavily represented by 'Unknown/Other' (which includes `zs`) (32.78%), Southeast Asia (32.78%), and East Asia (23.63%) language groups. This cluster clearly distinguishes users from Eastern regions. => From here, I think zs stands for China, but it looks weird to me.
    *   **`selected_template`:** Even distribution across most templates (B, D, E, F, G, H, J, K, L), with 'A' having a lower proportion, and a very small presence of 'C'.

*   **Insights:** This cluster highlights users from Eastern timezones with a reward rate slightly below average. The diverse `time_of_day` distribution suggests different engagement patterns. Tailoring content and offers to cultural contexts and preferences of these regions might significantly boost engagement. Further analysis on specific languages (`zs`, `vi`, `ja`, etc.) could uncover more granular insights.

---

#### Cluster 2: Morning/Night Engagers (Size: 35,399 samples)

*   **Key Characteristics:**
    *   **Average `n_eligible`:** ~9.44 (High number of eligible templates)
    *   **Average `local_hour`:** ~6.54 (Early Morning/Night in local time)
    *   **Average `timezone_offset`:** ~0.55 (Similar to Cluster 0, slightly positive offset from UTC)
    *   **`session_end_completed` Rate:** **13.63%** (Lower than global average)

*   **Categorical Distribution:**
    *   **`time_of_day_categorical`:** Dominated by Morning (63.93%) and Night (36.07%) local times. This cluster interacts very early or late in their local day.
    *   **`ui_language` & `timezone_group`:** Strong presence from UTC+0 (UK/Western Europe - ~48%) and UTC+1 (Western/Central Europe - ~35%) language groups, with a notable increase in UTC+2 (Eastern Europe/Middle East - ~9%).
    *   **`selected_template`:** Generally even distribution among templates (B, D, E, F, G, H, J, K, L), with 'A' having a lower proportion and no 'C' template.

*   **Insights:** This cluster interacts at unusual local hours compared to Cluster 0. The reward rate is also below average. This might represent users who check notifications first thing in the morning or late at night. Experimenting with different content types (e.g., summaries for morning, calming messages for night) or specific call-to-actions tailored to these times could improve engagement.

---

#### Cluster 3: High Reward/Specific Template 'C' Users (Size: 3,263 samples)

*   **Key Characteristics:**
    *   **Average `n_eligible`:** **~1.11** (Very low number of eligible templates, indicating specificity)
    *   **Average `local_hour`:** ~12.25 (Midday/Early Afternoon in local time)
    *   **Average `timezone_offset`:** ~0.67 (Similar positive offset to other European clusters)
    *   **`session_end_completed` Rate:** **40.45%** (Significantly higher than all other clusters and global average)

*   **Categorical Distribution:**
    *   **`time_of_day_categorical`:** Fairly balanced across Afternoon (32.33%), Evening (24.39%), Morning (24.27%), and Night (19.01%), but `local_hour` median is ~13, suggesting a focus around midday.
    *   **`ui_language` & `timezone_group`:** Similar representation from UTC+0 (UK/Western Europe - ~43%) and UTC+1 (Western/Central Europe - ~37%) language groups, but also more diverse small contributions from other regions.
    *   **`selected_template`:** **Overwhelmingly dominated by template 'C' (93.56%)**. Templates 'A', 'K', and 'H' account for very small proportions.

*   **Insights:** This is the most distinctive cluster, characterized by its exceptionally high success rate and highly specific behavior. Users in this cluster are almost exclusively targeted with template 'C', which clearly performs very well for them. This suggests that template 'C' is highly effective for this specific segment, likely due to its content, design, or the specific context of these users (e.g., specific user journey stage, specific type of user profile). The actionable insight here is to understand *why* template 'C' is so effective for this group and if its principles can be applied or adapted to other templates or user segments, or if this segment can be expanded. Furthermore, given the low `n_eligible`, these might be highly specific, valuable users with clear preferences.

# Visuals

**Reward Rate**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a bar plot to visualize the 'session_end_completed' rate for each cluster.
plt.figure(figsize=(8, 5))
sns.barplot(x=df['cluster'].unique(), y=df.groupby('cluster')['session_end_completed'].mean().values, palette='viridis', hue=df['cluster'].unique(), legend=False)
plt.title('Session End Completed Rate by Cluster')
plt.xlabel('Cluster')
plt.ylabel('Average Session End Completed Rate')
plt.ylim(0, 0.5) # Set a consistent y-limit for better comparison if rates are low
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

**Numerical features**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Generate comparative visualizations

features_to_visualize = ['n_eligible', 'local_hour']

plt.figure(figsize=(15, 5 * len(features_to_visualize)))

for i, feature in enumerate(features_to_visualize):
    plt.subplot(len(features_to_visualize), 1, i + 1)
    sns.boxplot(x='cluster', y=feature, data=df, palette='viridis', hue='cluster', legend=False)
    plt.title(f'Distribution of {feature.replace("_", " ").title()} by Cluster')
    plt.xlabel('Cluster')
    plt.ylabel(feature.replace("_", " ").title())
    plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

**Categorical features**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 5. Generate comparative visualizations (e.g., stacked bar charts or grouped bar plots of proportions)
# for key categorical features like 'timezone_group', the engineered 'time_of_day', and 'selected_template' across clusters.

categorical_features_to_visualize = ['timezone_group', 'time_of_day', 'selected_template']

plt.figure(figsize=(18, 6 * len(categorical_features_to_visualize)))

for i, feature in enumerate(categorical_features_to_visualize):
    plt.subplot(len(categorical_features_to_visualize), 1, i + 1)

    # Calculate proportions for stacked bar chart
    cluster_feature_counts = pd.crosstab(df['cluster'], df[feature])
    cluster_feature_proportions = cluster_feature_counts.div(cluster_feature_counts.sum(1), axis=0)

    cluster_feature_proportions.plot(kind='bar', stacked=True, ax=plt.gca(), cmap='viridis')
    plt.title(f'Proportion of {feature.replace("_", " ").title()} by Cluster')
    plt.xlabel('Cluster')
    plt.ylabel('Proportion')
    plt.xticks(rotation=0)
    plt.legend(title=feature.replace("_", " ").title(), bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()
